In [2]:
!pip install optuna
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.2/764.2 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 17.3 MB/s eta 0:00:00


In [8]:
import pandas as pd
import numpy as np

import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_squared_log_error

import optuna
import mlflow
import mlflow.lightgbm

import joblib


In [4]:
df = pd.read_parquet("/content/favorita_model_ready_2013_2015.parquet")

print(df.shape)
df.head()


(13913510, 27)


,id,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class,perishable,city,...,month,dayofweek,weekofyear,is_weekend,is_holiday,lag_7,lag_14,lag_28,rolling_7,rolling_14
0,1133118,2013-01-30,2,105574,3.0,0,GROCERY I,1045,0,Quito,...,1,2,5,0,0,3.0,4.0,21.0,4.857143,5.142857
1,1134542,2013-01-30,3,407499,16.0,0,EGGS,2502,1,Quito,...,1,2,5,0,0,14.0,17.0,19.0,16.571429,18.214286
2,1158346,2013-01-30,37,1066900,23.0,0,BEVERAGES,1136,0,Cuenca,...,1,2,5,0,0,20.0,21.0,89.0,23.142857,25.071429
3,1167420,2013-01-30,48,414752,10.0,0,GROCERY I,1072,0,Quito,...,1,2,5,0,0,9.0,5.0,19.0,11.000000,12.285714
4,1139592,2013-01-30,8,116017,19.0,0,GROCERY I,1072,0,Quito,...,1,2,5,0,0,8.0,5.0,22.0,19.000000,17.142857


## Feature definitions

In [5]:
FEATURES = [
    # identifiers
    "store_nbr",
    "item_nbr",

    # categorical structure
    "family",
    "class",
    "city",
    "cluster",

    # promotions & ops
    "onpromotion",
    "transactions",
    "perishable",

    # calendar
    "month",
    "dayofweek",
    "weekofyear",
    "is_weekend",

    # external signals
    "dcoilwtico",
    "is_holiday",

    # time-series features
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_7",
    "rolling_14"
]

CATEGORICAL_FEATURES = [
    "store_nbr",
    "item_nbr",
    "family",
    "class",
    "city",
    "cluster"
]

TARGET = "unit_sales"


In [6]:
for col in CATEGORICAL_FEATURES:
    df[col] = df[col].astype("category")


In [7]:
train_end = "2014-12-31"
valid_end = "2015-06-30"

train_df = df[df["date"] <= train_end]
valid_df = df[(df["date"] > train_end) & (df["date"] <= valid_end)]
test_df  = df[df["date"] > valid_end]

print("Train rows:", len(train_df))
print("Valid rows:", len(valid_df))
print("Test rows :", len(test_df))

Train rows: 8516559
Valid rows: 2431568
Test rows : 2965383


## Prep X, y

In [8]:
X_train = train_df[FEATURES]
X_valid = valid_df[FEATURES]
X_test  = test_df[FEATURES]

y_train_raw = np.clip(train_df[TARGET].values, 0, None)
y_valid_raw = np.clip(valid_df[TARGET].values, 0, None)
y_test_raw  = np.clip(test_df[TARGET].values, 0, None)

y_train_log = np.log1p(y_train_raw)
y_valid_log = np.log1p(y_valid_raw)
y_test_log  = np.log1p(y_test_raw)


## Smoke Test

In [10]:
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)


In [11]:
model.fit(
    X_train,
    y_train_log,
    categorical_feature=CATEGORICAL_FEATURES,
    eval_set=[(X_valid, y_valid_log)],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(30)]
)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.843541 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2777
[LightGBM] [Info] Number of data points in the train set: 8516559, number of used features: 20
[LightGBM] [Info] Start training from score 2.480552
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[273]	valid_0's rmse: 0.496109	valid_0's l2: 0.246124


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=500,
              n_jobs=-1, num_leaves=64, random_state=42, subsample=0.8)

In [12]:
y_test_pred_log = model.predict(
    X_test,
    num_iteration=model.best_iteration_
)

y_test_pred = np.expm1(y_test_pred_log)

test_rmsle = np.sqrt(
    mean_squared_log_error(
        y_test_raw,
        np.clip(y_test_pred, 0, None)
    )
)

test_rmsle


np.float64(0.5021530170650373)

In [14]:
from google.colab import files

model.booster_.save_model("favorita_lgb_baseline.txt")
files.download("favorita_lgb_baseline.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Rolling backtest

In [2]:
import pandas as pd
df = pd.read_parquet("/content/favorita_model_ready_2013_2015.parquet")


In [3]:
FEATURES = [
    "store_nbr", "item_nbr",
    "family", "class", "city", "cluster",
    "onpromotion", "transactions", "perishable",
    "month", "dayofweek", "weekofyear", "is_weekend",
    "dcoilwtico", "is_holiday",
    "lag_7", "lag_14", "lag_28",
    "rolling_7", "rolling_14"
]

CATEGORICAL_FEATURES = [
    "store_nbr", "item_nbr",
    "family", "class", "city", "cluster"
]

TARGET = "unit_sales"

for col in CATEGORICAL_FEATURES:
    df[col] = df[col].astype("category")


In [4]:
cutoff_dates = [
    "2014-06-30",
    "2014-09-30",
    "2014-12-31",
    "2015-03-31",
    "2015-06-30"
]

VAL_DAYS = 90    # ~3 months
TEST_DAYS = 90   # ~3 months


In [9]:
from datetime import timedelta
import numpy as np

results = []

for cutoff in cutoff_dates:
    cutoff = pd.to_datetime(cutoff)

    val_start = cutoff + timedelta(days=1)
    val_end   = val_start + timedelta(days=VAL_DAYS - 1)

    test_start = val_end + timedelta(days=1)
    test_end   = test_start + timedelta(days=TEST_DAYS - 1)

    # -----------------------------
    # Split data
    # -----------------------------
    train_df = df[df["date"] <= cutoff]
    val_df   = df[(df["date"] >= val_start) & (df["date"] <= val_end)]
    test_df  = df[(df["date"] >= test_start) & (df["date"] <= test_end)]

    if len(val_df) == 0 or len(test_df) == 0:
        continue

    X_train = train_df[FEATURES]
    X_val   = val_df[FEATURES]
    X_test  = test_df[FEATURES]

    y_train = np.log1p(np.clip(train_df[TARGET].values, 0, None))
    y_val   = np.log1p(np.clip(val_df[TARGET].values, 0, None))
    y_test_raw = np.clip(test_df[TARGET].values, 0, None)

    # -----------------------------
    # Train model
    # -----------------------------
    model = lgb.LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.05,
        num_leaves=64,
        min_data_in_leaf=200,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train,
        categorical_feature=CATEGORICAL_FEATURES,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(30)]
    )

    # -----------------------------
    # Evaluate on test
    # -----------------------------
    y_test_pred_log = model.predict(
        X_test,
        num_iteration=model.best_iteration_
    )

    y_test_pred = np.expm1(y_test_pred_log)

    rmsle = np.sqrt(
        mean_squared_log_error(
            y_test_raw,
            np.clip(y_test_pred, 0, None)
        )
    )

    results.append({
        "train_end": cutoff.date(),
        "val_start": val_start.date(),
        "val_end": val_end.date(),
        "test_start": test_start.date(),
        "test_end": test_end.date(),
        "rmsle": rmsle
    })

    print(f"Cutoff {cutoff.date()} → Test RMSLE: {rmsle:.4f}")


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.237239 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2688
[LightGBM] [Info] Number of data points in the train set: 5765614, number of used features: 20
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Info] Start training from score 2.444188
Training until validation scores don't improve for 30

In [10]:
results_df = pd.DataFrame(results)

results_df


,train_end,val_start,val_end,test_start,test_end,rmsle
0,2014-06-30,2014-07-01,2014-09-28,2014-09-29,2014-12-27,0.550607
1,2014-09-30,2014-10-01,2014-12-29,2014-12-30,2015-03-29,0.510651
2,2014-12-31,2015-01-01,2015-03-31,2015-04-01,2015-06-29,0.494618
3,2015-03-31,2015-04-01,2015-06-29,2015-06-30,2015-09-27,0.488175
4,2015-06-30,2015-07-01,2015-09-28,2015-09-29,2015-12-27,0.501323


In [11]:
mean_rmsle = results_df["rmsle"].mean()
std_rmsle  = results_df["rmsle"].std()

mean_rmsle, std_rmsle


(np.float64(0.5090747974171915), 0.024664298323747936)

In [1]:
import pandas as pd

# Recreate rolling backtest results
results_df = pd.DataFrame({
    "train_end": [
        "2014-06-30",
        "2014-09-30",
        "2014-12-31",
        "2015-03-31",
        "2015-06-30"
    ],
    "val_start": [
        "2014-07-01",
        "2014-10-01",
        "2015-01-01",
        "2015-04-01",
        "2015-07-01"
    ],
    "val_end": [
        "2014-09-28",
        "2014-12-29",
        "2015-03-31",
        "2015-06-29",
        "2015-09-28"
    ],
    "test_start": [
        "2014-09-29",
        "2014-12-30",
        "2015-04-01",
        "2015-06-30",
        "2015-09-29"
    ],
    "test_end": [
        "2014-12-27",
        "2015-03-29",
        "2015-06-29",
        "2015-09-27",
        "2015-12-27"
    ],
    "rmsle": [
        0.550607,
        0.510651,
        0.494618,
        0.488175,
        0.501323
    ]
})

# Convert date columns
date_cols = ["train_end", "val_start", "val_end", "test_start", "test_end"]
for col in date_cols:
    results_df[col] = pd.to_datetime(results_df[col])

results_df


,train_end,val_start,val_end,test_start,test_end,rmsle
0,2014-06-30,2014-07-01,2014-09-28,2014-09-29,2014-12-27,0.550607
1,2014-09-30,2014-10-01,2014-12-29,2014-12-30,2015-03-29,0.510651
2,2014-12-31,2015-01-01,2015-03-31,2015-04-01,2015-06-29,0.494618
3,2015-03-31,2015-04-01,2015-06-29,2015-06-30,2015-09-27,0.488175
4,2015-06-30,2015-07-01,2015-09-28,2015-09-29,2015-12-27,0.501323


In [2]:
import plotly.express as px

fig = px.line(
    results_df,
    x="train_end",
    y="rmsle",
    markers=True,
    title="Rolling Backtest Performance (RMSLE over Time)",
    labels={
        "train_end": "Training Window End Date",
        "rmsle": "Test RMSLE"
    }
)

fig.update_traces(marker=dict(size=10))
fig.update_layout(
    yaxis=dict(range=[0.47, 0.57]),
    template="plotly_white"
)

fig.show()


In [3]:
fig = px.bar(
    results_df,
    x="train_end",
    y="rmsle",
    title="RMSLE by Rolling Backtest Fold",
    labels={
        "train_end": "Training Window End Date",
        "rmsle": "Test RMSLE"
    },
    text="rmsle"
)

fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(
    yaxis=dict(range=[0.47, 0.57]),
    template="plotly_white"
)

fig.show()


In [4]:
mean_rmsle = results_df["rmsle"].mean()
std_rmsle = results_df["rmsle"].std()

fig = px.line(
    results_df,
    x="train_end",
    y="rmsle",
    markers=True,
    title="Rolling Backtest RMSLE with Mean and Variability"
)

fig.add_hline(
    y=mean_rmsle,
    line_dash="dash",
    annotation_text=f"Mean RMSLE = {mean_rmsle:.3f}",
    annotation_position="top left"
)

fig.add_hrect(
    y0=mean_rmsle - std_rmsle,
    y1=mean_rmsle + std_rmsle,
    fillcolor="gray",
    opacity=0.2,
    line_width=0
)

fig.update_layout(
    yaxis=dict(range=[0.47, 0.57]),
    template="plotly_white"
)

fig.show()


## P90 Quantile Model

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import mean_squared_log_error


In [3]:
df = pd.read_parquet("favorita_model_ready_2013_2015.parquet")


In [4]:
FEATURES = [
    "store_nbr", "item_nbr",
    "family", "class", "city", "cluster",
    "onpromotion", "transactions", "perishable",
    "month", "dayofweek", "weekofyear", "is_weekend",
    "dcoilwtico", "is_holiday",
    "lag_7", "lag_14", "lag_28",
    "rolling_7", "rolling_14"
]

CATEGORICAL_FEATURES = [
    "store_nbr", "item_nbr",
    "family", "class", "city", "cluster"
]

TARGET = "unit_sales"


In [5]:
for c in CATEGORICAL_FEATURES:
    df[c] = df[c].astype("category")


## Time-based Split

In [6]:
train_end = "2014-12-31"
valid_end = "2015-06-30"

train_df = df[df["date"] <= train_end]
valid_df = df[(df["date"] > train_end) & (df["date"] <= valid_end)]
test_df  = df[df["date"] > valid_end]

X_train = train_df[FEATURES]
X_valid = valid_df[FEATURES]
X_test  = test_df[FEATURES]


## Prepare target for quantile regression

In [7]:
y_train_raw = np.clip(train_df[TARGET].values, 0, None)
y_valid_raw = np.clip(valid_df[TARGET].values, 0, None)
y_test_raw  = np.clip(test_df[TARGET].values, 0, None)

y_train = np.log1p(y_train_raw)
y_valid = np.log1p(y_valid_raw)
y_test  = np.log1p(y_test_raw)


## Train the P90 Quantile Model

In [8]:
q90_model = lgb.LGBMRegressor(
    objective="quantile",
    alpha=0.90,                  # <-- P90
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=64,
    min_data_in_leaf=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

q90_model.fit(
    X_train,
    y_train,
    categorical_feature=CATEGORICAL_FEATURES,
    eval_set=[(X_valid, y_valid)],
    eval_metric="quantile",
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.803198 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2777
[LightGBM] [Info] Number of data points in the train set: 8516559, number of used features: 20
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Info] Start training from score 3.737670
Training until validation scores don't improve for 50

LGBMRegressor(alpha=0.9, colsample_bytree=0.8, learning_rate=0.03,
              min_data_in_leaf=200, n_estimators=3000, n_jobs=-1, num_leaves=64,
              objective='quantile', random_state=42, subsample=0.8)

In [9]:
pred_log = q90_model.predict(
    X_test,
    num_iteration=q90_model.best_iteration_
)

pred_p90 = np.expm1(pred_log)
order_qty = np.clip(pred_p90, 0, None)


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200


In [10]:
service_level = np.mean(y_test_raw <= order_qty)
service_level


np.float64(0.8746043934291119)

In [11]:
overage = np.maximum(order_qty - y_test_raw, 0)
underage = np.maximum(y_test_raw - order_qty, 0)

overage.mean(), underage.mean(), np.mean(underage > 0)


(np.float64(10.046733209680733),
 np.float64(1.0630656958033975),
 np.float64(0.12539560657088814))

In [12]:
rmsle = np.sqrt(
    mean_squared_log_error(
        y_test_raw,
        order_qty
    )
)

rmsle


np.float64(0.7432433409829707)

In [13]:
q90_model.booster_.save_model("favorita_lgb_q90.txt")


In [14]:
# During training (or after loading training df)
category_maps = {}

for col in CATEGORICAL_FEATURES:
    category_maps[col] = df[col].cat.categories

import joblib
joblib.dump(category_maps, "category_maps.pkl")


['category_maps.pkl']

In [16]:
import joblib

category_maps = {}
for col in CATEGORICAL_FEATURES:
    category_maps[col] = df[col].cat.categories

joblib.dump(category_maps, "category_maps.pkl")


['category_maps.pkl']